In [ ]:
import numpy as np

# 1. Corpus
corpus = "I love machine learning and I love deep learning"
corpus = corpus.lower().split()

# 2. Vocabulary
vocab = list(set(corpus))
vocab_size = len(vocab)

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

def one_hot(word):
    vector = np.zeros(vocab_size)
    vector[word_to_index[word]] = 1
    return vector


In [ ]:
# 3. Generate CBOW Training Data
window_size = 1
X = []
Y = []

for i in range(window_size, len(corpus) - window_size):
    context = []
    for j in range(-window_size, window_size + 1):
        if j != 0:
            context.append(one_hot(corpus[i + j]))

    target = one_hot(corpus[i])
    X.append(np.mean(context, axis=0))
    Y.append(target)

X = np.array(X)
Y = np.array(Y)


In [ ]:
# 4. Initialize Weights
embedding_size = 10
W1 = np.random.randn(vocab_size, embedding_size)
W2 = np.random.randn(embedding_size, vocab_size)

learning_rate = 0.01
epochs = 1000

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)



In [ ]:
# 5. Training
for epoch in range(epochs):
    loss = 0
    for i in range(len(X)):
        hidden = np.dot(X[i], W1)
        output = np.dot(hidden, W2)
        y_pred = softmax(output)

        loss += -np.sum(Y[i] * np.log(y_pred + 1e-9))

        error = y_pred - Y[i]
        dW2 = np.outer(hidden, error)
        dW1 = np.outer(X[i], np.dot(W2, error))

        W2 -= learning_rate * dW2
        W1 -= learning_rate * dW1

    if epoch % 200 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")



Epoch 0, Loss: 18.5710
Epoch 200, Loss: 1.9247
Epoch 400, Loss: 1.6820
Epoch 600, Loss: 1.6045
Epoch 800, Loss: 1.5667


In [ ]:
# 6. Word Embeddings
embeddings = W1

print("\nWord Embedding for 'love':")
print(embeddings[word_to_index["love"]])

# 7. Cosine Similarity Function
def cosine_similarity(word1, word2):
    vec1 = embeddings[word_to_index[word1]]
    vec2 = embeddings[word_to_index[word2]]
    similarity = np.dot(vec1, vec2) / (
        np.linalg.norm(vec1) * np.linalg.norm(vec2) + 1e-9)
    return similarity

# 8. Similarity Score
print("\nSimilarity between 'machine' and 'deep':")
print(cosine_similarity("machine", "deep"))

print("\nSimilarity between 'machine' and 'learning':")
print(cosine_similarity("machine", "learning"))


Word Embedding for 'love':
[ 0.20837719  0.90629145  1.03679415  1.2543919   1.79643371  0.6633123
  0.06831202 -1.21548403 -2.35159575 -1.65315584]

Similarity between 'machine' and 'deep':
-0.3357736835949987

Similarity between 'machine' and 'learning':
-0.09140154472014632


In [ ]:
# 9. Prediction Function
def predict_target(context_words):
    context_vectors = [one_hot(word) for word in context_words]
    context_mean = np.mean(context_vectors, axis=0)

    hidden = np.dot(context_mean, W1)
    output = np.dot(hidden, W2)
    y_pred = softmax(output)

    return index_to_word[np.argmax(y_pred)]

context = ["love", "learning"]
print("\nGiven Context:", context)
print("Predicted Target Word:", predict_target(context))


Given Context: ['love', 'learning']
Predicted Target Word: deep
